# SimSwap
Reference: https://github.com/neuralchen/SimSwap

## Prepare code

In [ ]:
%cd /content
!git clone https://github.com/woctezuma/SimSwap.git
%cd /content/SimSwap/
!git checkout upgrade-insightface

In [ ]:
!pip install insightface==0.7.3 onnxruntime moviepy > /dev/null
!pip install imageio==2.34.0 > /dev/null

## Prepare models

In [ ]:
%cd /content/SimSwap

In [ ]:
!wget https://github.com/woctezuma/SimSwap-colab/releases/download/antelope/antelope.zip
!wget -P ./arcface_model https://github.com/woctezuma/SimSwap-colab/releases/download/1.0/arcface_checkpoint.tar
!wget https://github.com/neuralchen/SimSwap/releases/download/1.0/checkpoints.zip
!wget -P ./parsing_model/checkpoint https://github.com/neuralchen/SimSwap/releases/download/1.0/79999_iter.pth
!wget https://github.com/neuralchen/SimSwap/releases/download/512_beta/512.zip

In [ ]:
!unzip ./checkpoints.zip  -d ./checkpoints

!unzip 512.zip -d ./checkpoints

!unzip antelope.zip -d ./insightface_func/models/

## Prepare data

### Download

In [ ]:
%cd /content

!curl https://i.imgur.com/QYJOzy7.jpeg -o cpc_ackboo.jpg
!curl https://i.imgur.com/l5MGOws.jpeg -o starwars_meme.jpg

In [ ]:
from PIL import Image
def get_new_size(img_size,
                 max_allowed_length = 512):

  if any(max_allowed_length < sz for sz in img_size):
    long_length = max(img_size)
    ratio = max_allowed_length / long_length
  else:
    ratio = 1.0

  new_img_size = [
                  int(ratio*sz)
                  for sz in img_size
                  ]

  return tuple(new_img_size)

img_name = [11,111,14,16,18,19,2,3,41,5,51,54,56,62,8,9]
for idx in img_name:
  input_fname = f'/content/source/{idx}.png'
  output_fname = f'/content/target/{idx}.png'




  allow_resize = True

  for fname in [input_fname, output_fname]:
    jpg_fname = fname.replace('.png', '.jpg')

    try:
      img = Image.open(fname)
    except FileNotFoundError:
      continue

    new_size = get_new_size(img.size, max_allowed_length = 512)
    if allow_resize:
      print(f'Resizing from {img.size} to {new_size}')
      img =img.resize(new_size)

    print(f'Saving to {jpg_fname}')
    img.convert('RGB').save(jpg_fname)

  jpg_input = input_fname.replace('.png', '.jpg')
  jpg_output = output_fname.replace('.png', '.jpg')




In [ ]:
%cd /content/SimSwap
%mkdir -p /content/output/single/

!python test_wholeimage_swapsingle.py \
 --no_simswaplogo \
 --use_mask \
 --crop_size 512 \
 --isTrain false  --name people \
 --Arc_path arcface_model/arcface_checkpoint.tar \
 --pic_a_path {jpg_input} \
 --pic_b_path {jpg_output} \
 --output_path /content/output/single/ > /dev/null

In [ ]:
import os
import subprocess
from PIL import Image

# 调整图像尺寸的函数
def get_new_size(img_size, max_allowed_length=512):
    if any(max_allowed_length < sz for sz in img_size):
        long_length = max(img_size)
        ratio = max_allowed_length / long_length
    else:
        ratio = 1.0

    new_img_size = [
        int(ratio*sz)
        for sz in img_size
    ]

    return tuple(new_img_size)

# 需要处理的图片ID列表
img_name = [11, 111, 14, 16, 18, 19, 2, 3, 41, 5, 51, 54, 56, 62, 8, 9]

# 设置输出目录
output_dir = '/content/output/single/'
os.makedirs(output_dir, exist_ok=True)

# 遍历所有需要处理的图片
for idx in img_name:
    print(f"\n===== 处理图片 {idx} =====")

    input_fname = f'/content/source/{idx}.png'
    output_fname = f'/content/target/{idx}.png'

    allow_resize = True

    # 预处理源图像和目标图像
    for fname in [input_fname, output_fname]:
        jpg_fname = fname.replace('.png', '.jpg')

        try:
            img = Image.open(fname)
        except FileNotFoundError:
            print(f"文件不存在: {fname}")
            continue

        new_size = get_new_size(img.size, max_allowed_length=512)
        if allow_resize:
            print(f'调整大小: {img.size} -> {new_size}')
            img = img.resize(new_size)

        print(f'保存到: {jpg_fname}')
        img.convert('RGB').save(jpg_fname)

    jpg_input = input_fname.replace('.png', '.jpg')
    jpg_output = output_fname.replace('.png', '.jpg')

    # 执行人脸交换
    cmd = [
        "python", "test_wholeimage_swapsingle.py",
        "--no_simswaplogo",
        "--use_mask",
        "--crop_size", "512",
        "--isTrain", "false",
        "--name", "people",
        "--Arc_path", "arcface_model/arcface_checkpoint.tar",
        "--pic_a_path", jpg_input,
        "--pic_b_path", jpg_output,
        "--output_path", output_dir
    ]

    print("执行命令:", " ".join(cmd))
    result = subprocess.run(cmd)

    # 如果生成了结果，重命名以避免覆盖
    result_path = os.path.join(output_dir, 'result.jpg')
    if os.path.exists(result_path):
        new_result_path = os.path.join(output_dir, f'result_{idx}.jpg')
        os.rename(result_path, new_result_path)
        print(f"结果已保存到: {new_result_path}")
    else:
        print(f"警告: 处理图片 {idx} 未生成结果文件")

print("\n所有图片处理完成!")

### Convert to JPG

Images should not be too large, hence the (arbitrary) limitation of 1024 length.

In [ ]:
def get_new_size(img_size,
                 max_allowed_length = 1024):

  if any(max_allowed_length < sz for sz in img_size):
    long_length = max(img_size)
    ratio = max_allowed_length / long_length
  else:
    ratio = 1.0

  new_img_size = [
                  int(ratio*sz)
                  for sz in img_size
                  ]

  return tuple(new_img_size)

In [ ]:
from PIL import Image

allow_resize = False

for fname in [input_fname, output_fname]:
  jpg_fname = fname.replace('.png', '.jpg')

  try:
    img = Image.open(fname)
  except FileNotFoundError:
    continue

  new_size = get_new_size(img.size, max_allowed_length = 1024)
  if allow_resize:
    print(f'Resizing from {img.size} to {new_size}')
    img.resize(new_size)

  print(f'Saving to {jpg_fname}')
  img.convert('RGB').save(jpg_fname)

jpg_input = input_fname.replace('.png', '.jpg')
jpg_output = output_fname.replace('.png', '.jpg')

## Run

In [ ]:
!python test.py

### Single

In [ ]:
%cd /content/SimSwap
%mkdir -p /content/output/single/

!python test_wholeimage_swapsingle.py \
 --no_simswaplogo \
 --use_mask \
 --crop_size 512 \
 --isTrain false  --name people \
 --Arc_path arcface_model/arcface_checkpoint.tar \
 --pic_a_path {jpg_input} \
 --pic_b_path {jpg_output} \
 --output_path /content/output/single/ > /dev/null


### Multi

In [ ]:
%cd /content/SimSwap
%mkdir -p /content/output/multi/

!python test_wholeimage_swapmulti.py \
 --no_simswaplogo \
 --use_mask \
 --crop_size 512 \
 --isTrain false  --name people \
 --Arc_path arcface_model/arcface_checkpoint.tar \
 --pic_a_path {jpg_input} \
 --pic_b_path {jpg_output} \
 --output_path /content/output/multi/ > /dev/null
